# RAG Pipeline - Groq Edition

Same pipeline as `rag_pipeline_walkthrough.ipynb` but uses **Groq API** instead of Ollama.

- Embeddings, ChromaDB, Reranking: unchanged (local, no Ollama needed)
- LLM calls: Groq cloud API
- No existing source files are modified

**When to use:** CPU-only machine, or when Ollama is too slow.

---
## Step 0 - Install Groq SDK (run once)

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', 'langchain-groq', '-q'])
print('Done.')

---
## Step 1 - Setup: Paths + Groq API Key

In [ ]:
import os, sys

NOTEBOOK_DIR = os.path.abspath('')
REPO_ROOT    = os.path.dirname(NOTEBOOK_DIR)
SRC_PATH     = os.path.join(REPO_ROOT, 'src')
CHROMA_PATH  = os.path.join(REPO_ROOT, 'vectordb', 'chroma_db')

if SRC_PATH not in sys.path:
    sys.path.insert(0, SRC_PATH)

print('SRC_PATH   :', SRC_PATH)
print('CHROMA_PATH:', CHROMA_PATH)

# Paste your free Groq API key here (get it from https://console.groq.com)
GROQ_API_KEY = ''   # <-- paste your gsk_... key between the quotes

if not GROQ_API_KEY:
    raise ValueError(
        'GROQ_API_KEY is empty. '
        'Get a free key at https://console.groq.com -> API Keys -> Create key'
    )

os.environ['GROQ_API_KEY'] = GROQ_API_KEY
print('Groq API key set.')

---
## Step 2 - Create the Groq LLM (replaces Ollama)

This is the **only difference** from the Ollama notebook. `ChatGroq` is a drop-in replacement for `ChatOllama`.

In [ ]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model='llama-3.1-8b-instant',
    temperature=0,
    api_key=GROQ_API_KEY
)

test_response = llm.invoke('Reply with exactly: Groq is working.')
print(test_response.content)

---
## Step 3 - Load Dataset (RAGBench Finance)

In [ ]:
from datasets import load_dataset

ds = load_dataset('rungalileo/ragbench', 'finqa', split='test')

print(f'Total rows : {len(ds)}')
print(f'Columns    : {ds.column_names}')

sample = ds[0]
print(f'Question   : {sample["question"]}')
print(f'Gold Answer: {sample["response"]}')

---
## Step 4 - Load Embedder (local, no Groq needed)

In [ ]:
from embeddings.embedder import get_embedder

embedder = get_embedder()

test_vec = embedder.encode('rate of return on investment')
print(f'Embedding shape : {test_vec.shape}')
print(f'First 5 values  : {test_vec[:5].round(4)}')

---
## Step 5 - Connect to ChromaDB (existing finance collection)

In [ ]:
import chromadb

client = chromadb.PersistentClient(path=CHROMA_PATH)
collection = client.get_collection(name='finance')

print(f'Collections    : {[c.name for c in client.list_collections()]}')
print(f'Finance chunks : {collection.count()}')

---
## Step 6 - Retrieve Chunks for a Query

In [ ]:
sample = ds[0]
query  = sample['question']
gold   = sample['response']

print(f'Query      : {query}')
print(f'Gold Answer: {gold}')

query_vector = embedder.encode(query).tolist()

results = collection.query(
    query_embeddings=[query_vector],
    n_results=10
)

print(f'Retrieved {len(results["documents"][0])} chunks.')
for i, doc in enumerate(results['documents'][0][:3]):
    print(f'  [{i+1}] {doc[:120]}...')

---
## Step 7 - Rerank (local cross-encoder, no Groq)

In [ ]:
from reranking.reranker import rerank

reranked = rerank(query, results)[:3]

for i, chunk in enumerate(reranked):
    print(f'--- Chunk {i+1} ---')
    print(chunk[:200])
    print()

---
## Step 8 - Generate Answer with Groq LLM

In [ ]:
from prompts.answer_prompt import ANSWER_PROMPT

def generate_answer_groq(query: str, context_chunks: list) -> str:
    context  = '\n\n'.join(context_chunks)
    prompt   = ANSWER_PROMPT.format(query=query, context=context)
    response = llm.invoke(prompt)
    return response.content.strip()

answer = generate_answer_groq(query, reranked)

print('Generated Answer:')
print(answer)
print(f'Gold Answer: {gold}')

---
## Step 9 - Evaluate with Groq LLM as Judge

**Fix applied here:** `reasoning` is `Optional` in the local schema.
Groq llama-3.1-8b sometimes skips the reasoning field causing a `BadRequestError`.
Making it optional prevents the error without touching any source files.

In [ ]:
from pydantic import BaseModel, Field
from typing import Optional
from prompts.evaluation_prompt import RAG_EVALUATION_PROMPT

# Local schema with reasoning Optional - fixes Groq BadRequestError
class RAGEvaluationCountsGroq(BaseModel):
    total_chunks: int                = Field(description='Total retrieved chunks')
    relevant_chunks: int             = Field(description='Chunks relevant to the question')
    total_context_facts: int         = Field(description='Total important facts in context')
    used_context_facts: int          = Field(description='Context facts used in the answer')
    total_relevant_facts: int        = Field(description='Relevant facts needed to answer')
    covered_relevant_facts: int      = Field(description='Relevant facts covered in answer')
    total_answer_statements: int     = Field(description='Total factual statements in answer')
    supported_answer_statements: int = Field(description='Statements supported by context')
    reasoning: Optional[str]         = Field(default='', description='Short evaluation explanation')

def evaluate_groq(question: str, contexts: list, answer: str) -> RAGEvaluationCountsGroq:
    prompt = RAG_EVALUATION_PROMPT.format(
        question=question,
        contexts='\n\n'.join(contexts),
        answer=answer
    )
    judge = llm.with_structured_output(RAGEvaluationCountsGroq)
    return judge.invoke(prompt)

def calculate_metrics(result: RAGEvaluationCountsGroq) -> dict:
    return {
        'context_relevance':   result.relevant_chunks             / max(result.total_chunks, 1),
        'context_utilization': result.used_context_facts          / max(result.total_context_facts, 1),
        'completeness':        result.covered_relevant_facts      / max(result.total_relevant_facts, 1),
        'adherence':           result.supported_answer_statements / max(result.total_answer_statements, 1),
    }

eval_result = evaluate_groq(query, reranked, answer)
metrics     = calculate_metrics(eval_result)

print('RAG Evaluation Results (single question):')
print(f'  Context Relevance   : {metrics["context_relevance"]:.2f}')
print(f'  Context Utilization : {metrics["context_utilization"]:.2f}')
print(f'  Completeness        : {metrics["completeness"]:.2f}')
print(f'  Adherence           : {metrics["adherence"]:.2f}')
if eval_result.reasoning:
    print(f'Reasoning: {eval_result.reasoning}')

---
## Step 10 - Benchmark: Evaluate N Questions

Change `NUM_SAMPLES` to however many questions you want.
With Groq this finishes in **2-4 minutes** for 10 questions.

In [ ]:
import time

NUM_SAMPLES = 10

results_log = []
start_time  = time.time()

for i, row in enumerate(ds.select(range(NUM_SAMPLES))):
    q    = row['question']
    gold = row['response']
    print(f'[{i+1}/{NUM_SAMPLES}] {q[:70]}...')

    qvec   = embedder.encode(q).tolist()
    raw    = collection.query(query_embeddings=[qvec], n_results=10)
    chunks = rerank(q, raw)[:3]

    ans = generate_answer_groq(q, chunks)

    ev  = evaluate_groq(q, chunks, ans)
    m   = calculate_metrics(ev)
    m['question'] = q
    m['answer']   = ans
    m['gold']     = gold
    results_log.append(m)

    print(f'   CR:{m["context_relevance"]:.2f}  CU:{m["context_utilization"]:.2f}  CO:{m["completeness"]:.2f}  AD:{m["adherence"]:.2f}')

elapsed = time.time() - start_time
print(f'AVERAGE OVER {NUM_SAMPLES} QUESTIONS  (took {elapsed:.0f}s total)')
for metric in ['context_relevance', 'context_utilization', 'completeness', 'adherence']:
    avg = sum(r[metric] for r in results_log) / len(results_log)
    print(f'  {metric:<25}: {avg:.4f}')

---
## Step 11 - Save Results to CSV (Optional)

In [ ]:
import csv

out_dir  = os.path.join(REPO_ROOT, 'benchmarks')
out_file = os.path.join(out_dir, 'groq_results.csv')
os.makedirs(out_dir, exist_ok=True)

fields = ['question', 'gold', 'answer',
          'context_relevance', 'context_utilization', 'completeness', 'adherence']

with open(out_file, 'w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=fields)
    writer.writeheader()
    for row in results_log:
        writer.writerow({k: row[k] for k in fields})

print(f'Saved {len(results_log)} rows to {out_file}')

---
## Quick Reference

| Component | Runs where |
|---|---|
| Embeddings (BAAI/bge-small-en-v1.5) | Local CPU |
| ChromaDB vector search | Local disk |
| Reranker (BAAI/bge-reranker-base) | Local CPU |
| LLM - Answer Generation | Groq API (cloud) |
| LLM - Evaluation Judge | Groq API (cloud) |

Groq free tier: llama-3.1-8b-instant gives 500,000 tokens/day.
10 questions uses ~40,000 tokens - well within free limits.

To switch to a larger model, in Step 2 change the model name:
```python
llm = ChatGroq(model='llama-3.3-70b-versatile', temperature=0)  # smarter
llm = ChatGroq(model='llama-3.1-8b-instant',    temperature=0)  # fast (default)
```